# env_functions — hub dos módulos do pipeline

Este notebook não guarda código: ele carrega, em ordem, os módulos de
`modulos/` (cada um self-contained, com seus próprios imports e documentação):

- **`./modulos/deteccao`** — detecção automática de abas e colunas de identidade (multi-edition):
  `detect_munic_sheets`, `detect_identity_renames`;
- **`./modulos/upsert`** — gravação/upsert de tabelas Delta particionadas por `ano_edicao`:
  `upsert_delta_table` e os 6 passos auxiliares (o `_set_user_metadata` daqui é
  reutilizado pelo ledger em tempo de chamada);
- **`./modulos/ledger`** — o ledger do pipeline (`pipeline_manifest`, docs/arquitetura):
  `gravar_linha_manifest`, `contrato_da_edicao`, `check_igual`, `resumo_validacoes`,
  `verificar_arquivo_landing` — e a medição do shape da edição: `colunas_com_dados`,
  `celulas_recusa`.

Os consumidores não mudam: continua sendo `%run ../environment/env_functions`.
A ordem dos %runs importa (deteccao → upsert → ledger): o ledger chama
`_set_user_metadata`, definido no upsert, apenas em tempo de chamada.
Os overrides de exceção e as variáveis de configuração seguem no `env_variables`.

## Convenções de nomenclatura dos módulos

**API pública × interna:** os notebooks consumidores só chamam funções **sem** prefixo. Funções com prefixo `_` (ex.: `_nome_canonico`, `_eh_cod_municipio`, `_adicionar_ano_edicao_na_origem`) são internas do módulo — só a função pública dele as chama; não fazem parte do contrato do hub. O `_` é a convenção Python para isso (IDEs, linters e qualquer pessoa que leia Python a reconhecem); um prefixo tipo `aux_` não diria nada sobre o que a função faz — o que a descreve é o nome depois do prefixo.

**Taxonomia de verbos (o que cada padrão promete):**

| Padrão | Semântica | Exemplos |
| --- | --- | --- |
| `gravar_*` | escreve/atualiza a linha da etapa no ledger (MERGE na chave) | `gravar_linha_manifest` |
| `contrato_*` | lê o contrato da edição; aborta com instrução se ausente/desmentido | `contrato_da_edicao` |
| `check_*` | **registra** um check PASS/FAIL (obtido vs. esperado do contrato); **não interrompe** — acumula no lote (lote = o bloco de checks do notebook: todos os `check_*` até o último `resumo_*`, que o encerra) | `check_igual` |
| `resumo_*` | **fim do lote**: imprime o placar (ex.: "3/4 checks PASS") e, se algum FAIL, interrompe o run INTEIRO com todas as falhas listadas de uma vez (fail-fast em bloco) — escritas já commitadas NÃO são desfeitas, por isso o lote fecha antes da primeira escrita da etapa; a lista esvazia aqui, um lote novo começa depois | `resumo_validacoes` |
| `verificar_*` | confere e **avisa** (AVISO no output, sem abortar) | `verificar_arquivo_landing` |
| `detect_*` | detecção automática por nome canônico; aborta se ambíguo (fail-fast) | `detect_munic_sheets`, `detect_identity_renames` |

Medição do shape (`colunas_com_dados`, `celulas_recusa`) e gravação (`upsert_delta_table`) são operações pontuais com nome próprio — a tabela cobre os padrões recorrentes. Nos notebooks, o lote termina num único `resumo_validacoes()` **antes da primeira escrita** da etapa: um run reprovado morre com todas as falhas de uma vez, sem deixar a tabela meio-escrita.

**Promoção de função de notebook para módulo:** função definida dentro de um notebook do pipeline fica **local** enquanto tiver um único consumidor — ex.: `extract_var_code_meaning`/`obtain_filled_cells`, `prepare_data_to_delta_format` e `add_ingestion_metadata` (landing_to_bronze). Promove para `modulos/` (via hub) só quando (a) aparecer um **segundo consumitor**, ou (b) se formar uma **família** de funções do mesmo tema (ex.: detecção de abas + colunas → `deteccao`). No dia da promoção, o módulo novo nasce self-contained como os existentes, seus helpers internos ganham `_` (passam a ser internos do módulo) e os notebooks consumidores continuam chamando pelo caminho de sempre (`%run ../environment/env_functions`).

In [0]:
%run ./modulos/deteccao

In [0]:
%run ./modulos/upsert

In [0]:
%run ./modulos/ledger